# Pipeline de datos con tf.data para PlantVillage

Este notebook contiene funciones para escanear el dataset, asignar etiquetas binarias, y construir pipelines de datos optimizados para entrenamiento, validación y prueba en TensorFlow.

## 1. Importar librerías necesarias

In [ ]:
from pathlib import Path
from typing import Callable, List, Tuple
import tensorflow as tf

## 2. Definir extensiones soportadas y función para escanear el dataset

In [ ]:
# Extensiones de imagen soportadas
IMAGE_EXTENSIONS = ["*.jpg", "*.JPG", "*.jpeg", "*.JPEG", "*.png", "*.PNG"]

def scan_plant_village(data_dir: str) -> Tuple[List[str], List[int]]:
    """
    Escanea un directorio estilo PlantVillage y asigna labels binarios.
    Si 'healthy' está en el nombre de la carpeta, label=0; si no, label=1.
    """
    data_path = Path(data_dir)
    if not data_path.exists():
        raise FileNotFoundError(f"Directorio no encontrado: {data_dir}")

    image_paths = []
    labels = []
    for class_dir in sorted(data_path.iterdir()):
        if not class_dir.is_dir():
            continue
        label = 0 if "healthy" in class_dir.name.lower() else 1
        for ext in IMAGE_EXTENSIONS:
            for img_path in class_dir.glob(ext):
                image_paths.append(str(img_path))
                labels.append(label)
    print(f"Escaneadas {len(image_paths)} imágenes: healthy={labels.count(0)}, diseased={labels.count(1)}")
    return image_paths, labels

## 3. Capa de data augmentation para entrenamiento

In [ ]:
def build_augmentation_layer() -> tf.keras.Sequential:
    """
    Capa de augmentation para entrenamiento.
    ⚠️ Aplicar SOLO al conjunto de train. Val y test sin augmentation.
    """
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.10, 0.10),
        tf.keras.layers.RandomBrightness(factor=0.2),
    ], name="augmentation")

## 4. Función para cargar y preprocesar imágenes

In [ ]:
def _load_image(image_path: tf.Tensor, label: tf.Tensor, image_size: Tuple[int, int], preprocess_fn: Callable):
    """Carga, redimensiona y preprocesa una imagen."""
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, image_size)
    img = preprocess_fn(img)
    return img, tf.cast(label, tf.float32)

## 5. Construir el pipeline tf.data

In [ ]:
def build_dataset(
    image_paths: List[str],
    labels: List[int],
    image_size: Tuple[int, int],
    preprocess_fn: Callable,
    batch_size: int = 32,
    augment: bool = False,
    shuffle: bool = False,
    seed: int = 42,
) -> tf.data.Dataset:
    """
    Construye un pipeline tf.data optimizado.
    Args:
        image_paths: Rutas a las imágenes
        labels: Labels binarios (0/1)
        image_size: (height, width)
        preprocess_fn: preprocess_input del modelo
        batch_size: Tamaño de batch
        augment: True SOLO para train
        shuffle: True SOLO para train
        seed: Semilla para shuffle
    Returns:
        tf.data.Dataset optimizado con prefetch
    """
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda path, label: _load_image(path, label, image_size, preprocess_fn),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(batch_size, drop_remainder=False)
    if augment:
        aug_layer = build_augmentation_layer()
        ds = ds.map(
            lambda x, y: (aug_layer(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    return ds.prefetch(tf.data.AUTOTUNE)